In [ ]:
from selenium import webdriver # 크롤러 역할
from selenium.webdriver.common.by import By # 크롤러가 어떤 방식으로 값을 찾아오게 할 것인지 결정
from selenium.webdriver.chrome.service import Service # 크롤러에게 로컬컴퓨터 내 크롬브라우저 드라이브 일치 & 설치 지원
from selenium.webdriver.chrome.options import Options # 크롤러가 어떤 환경에서 어떤 방식으로 크롤링 하도록 할지 옵션 설정
from webdriver_manager.chrome import ChromeDriverManager # 실제 드라이브 설치 역할

import pandas as pd
import time # 특정 사이트 접속 후 스크립트 기반의 코드를 서버로부터 가져올 때까지 시간을 벌어주는 역할
import re # regular expression
service = Service(ChromeDriverManager().install())
options = Options()

options.add_argument("--headless") # 가상브라우저를 현재 내 로컬컴퓨터에 나타나지 않도록 할 때
options.add_argument("--disable-gpu") # 가상브라우저를 사용하지 않기 때문에 굳이 gpu 메모리를 사용하지 않겠다는 의미
options.add_argument("--disable-dev-shm-usage") # 큰 데이터의 크롤링 시, 데이터를 개발자도구 내 임시메모리 공간을 사용 x (로컬공간) 
options.add_argument("--window-size=1920,1080") # 크롤러의 웹 브라우저 사이즈 세팅
options.add_argument("--start-maximized") # 주변 환경요소와 무관하게 브라우저 사이즈를 무조건 크게
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36") # 브라우저 식별 정보 = 우리가 정상적인 웹사이트 방문자
options.add_argument("--lang=ko_KR") # 접속 및 방문하 브라우저의 기본세팅 언어
options.add_argument("--no-sandbox") # 규제 적용 방지

URL = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(URL)
time.sleep(3)

# 유틸리티 함수
def to_int(text) :
    num = re.sub(r"[^0-9]", "", text)
    return int(num) if num else 0

# 베스트 버튼 클릭
best_button = driver.find_element(By.XPATH, "//span[text()='베스트']")
best_button.click()
time.sleep(3)

# 상품카드 정보 크롤링
items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print(f"1. ✅ 찾은 상품 수: {len(items)}개")

# 상품 수집데이터 저장용 리스트 생성
products = list() # []

# 상품별 상세데이터 수집
for idx, item in enumerate(items, 1) :
    # 상품 랭킹 번호 수집
    try :
        ranking_no = int(item.find_element(By.CSS_SELECTOR, "span.rank").text)
    except :
        ranking_no = None

    # 브랜드 이름 수집
    try :
        brand_name = item.find_element(By.CSS_SELECTOR, "span.text.title").text.strip()
    except :
        brand_name = None

    # 상품 이름 수집
    try :
        product_name = item.find_element(By.CSS_SELECTOR, "span.text.detail").text.strip()
    except :
        product_name = None
    
    # 판매가격 (*소비자가, 할인가, 할인률) 수집
    price_box = item.find_element(By.CSS_SELECTOR, "span.prdc-price")

    # 소비자가
    try :
        original_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.customer-price").text)
    except :
        original_price = None

    # 할인가
    try :
        sale_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-price").text)
    except :
        sale_price = None
        
    # 할인률
    try :
        discount_rate = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-discount").text)
    except :
        discount_rate = 0

    try :
        stats = item.find_element(By.CSS_SELECTOR, "span.stats")
    
        # 평점
        try :
            rating = float(stats.find_element(By.CSS_SELECTOR, "span.review em.score").text)
        except :
            rating = None
    
        # 리뷰수
        try :
            review_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.review span.text.cnt").text)
        except :
            review_count = 0
    
    
        # 좋아요수
        try :
            like_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.like span.text.cnt").text)
        except :
            like_count = 0
    
    except :
        rating = None
        review_count = 0
        like_count = 0

    # 상품상세페이지 url
    img_url = item.find_element(By.CSS_SELECTOR, "span.img img").get_attribute("src")

    match = re.search(r"/(\d{9})[_.]", img_url)

    if match :
        product_id = match.group(1)
        product_url = f"https://www.wconcept.co.kr/Product/{product_id}"
    else :
        product_id = None
        product_url = None
            
    products.append({
        "ranking_no": ranking_no,
        "brand_name": brand_name,
        "product_name": product_name,
        "original_price": original_price,
        "sale_price": sale_price,
        "discount_rate": discount_rate,
        "rating": rating,
        "review_count": review_count,
        "like_count": like_count,
        "product_id": product_id,
        "product_url": product_url,
        "img_url": img_url,
        "category_depth1": None,
        "category_depth2": None,
        "category_depth3": None,
        "category_depth4": None
    })
   
print("2. ✅ wconcept 상품기본 정보수집 완료!")

# 각 상품별 리뷰수집 데이터 저장용 리스트
all_reviews = list() # []

#for product in products[:1]
for product in products :

    product_url = product["product_url"]

    # 상품 상세페이지 진입단계
    # --------------------------------------------------------
    # 상품 URL이 없으면 상세페이지 접근 불가
    # --------------------------------------------------------

    if product_url is None:
        print("상품 URL 없음 → SKIP")
        continue

    # --------------------------------------------------------
    # 상품 상세페이지 진입
    # --------------------------------------------------------

    driver.get(product_url)
    time.sleep(10)

    # --------------------------------------------------------
    # 카테고리 수집
    # --------------------------------------------------------
    
    try:
        category_elements = driver.find_elements(
            By.CSS_SELECTOR,
            "#container > .pdt > #prdLocaiton > li > a, #container > .pdt > #prdLocaiton > li > button"
        )
    
        category_list = []
    
        for element in category_elements:
            text = element.text.strip()
    
            if text:
                category_list.append(text)
    
        # HOME 제거
        if category_list and category_list[0] == "HOME":
            category_list = category_list[1:]
    
        print("수집된 카테고리 :", category_list)
    
    except Exception as e:
    
        print("카테고리 수집 실패 :", e)
    
        category_list = []

    # --------------------------------------------------------
    # depth 별 분리
    # --------------------------------------------------------

    product["category_depth1"] = category_list[0] if len(category_list) > 0 else None
    product["category_depth2"] = category_list[1] if len(category_list) > 1 else None
    product["category_depth3"] = category_list[2] if len(category_list) > 2 else None
    product["category_depth4"] = category_list[3] if len(category_list) > 3 else None

    # --------------------------------------------------------
    # 리뷰가 없는 상품이면 카테고리만 저장하고 다음 상품으로
    # --------------------------------------------------------

    if product["review_count"] == 0 :
        print("리뷰 없음 → 카테고리만 수집")
        continue

    try : 
        review_button = driver.find_element(By.CSS_SELECTOR, "#reviewCnt1")
        review_button.click()
    
        time.sleep(3)
        # print("2-1. ✅ 리뷰버튼 클릭 완료!")
    except :
        # print("2-2. ✅ 리뷰버튼이 없는 상품입니다 -> SKIP")
        continue

    print("3. ✅ wconcept 상품상세페이지 진입 완료!")
    for page_no in range(1, 4) :
        if page_no > 1 :
            page_buttons = driver.find_elements(By.CSS_SELECTOR, f"ul#reviewPageNavigation a[title='{page_no}']")

            if len(page_buttons) == 0 :
                print(f"{page_no}페이지가 없습니다.")
                print("리뷰 수집을 종료합니다.")

                break

            page_buttons[0].click()

            time.sleep(2)
                
        rows = driver.find_elements(By.CSS_SELECTOR, "tr")

        # 진짜 리뷰 정보값을 가지고 있는 tr만 저장할 목적의 리스트
        review_items = list()
    
        for row in rows :
            review_text_elements = row.find_elements(By.CSS_SELECTOR, "p.pdt_review_text")
    
            if len(review_text_elements) > 0 :
                review_items.append(row)

        print(f"{page_no}) 페이지 리뷰 수: {len(review_items)}개")
    
        for review_index, review in enumerate(review_items, 1) :

            review_no = (page_no - 1) * 6 + review_index
            
            # 리뷰 작성자
            reviewer = review.find_element(By.CSS_SELECTOR, "p.product_review_info_right > em").text.strip()
    
            # 리뷰 작성일
            review_date = review.find_element(By.CSS_SELECTOR, "p.product_review_info_right > span").text.strip()
    
            # 구매 옵션정보
            try :
                purchase_text = review.find_element(By.CSS_SELECTOR, "div.pdt_review_option span").text.strip()
                purchase_option = re.sub(r"^구매옵션\s*:\s*", "", purchase_text)
                purchase_option = purchase_option.rstrip(",").strip()

                if purchase_option == "" :
                    purchase_option = None
            except :
                purchase_option = None
    
            # 리뷰 본문
            review_text = review.find_element(By.CSS_SELECTOR, "p.pdt_review_text").text.strip()
            review_text = re.sub(r"\s+", " ", review_text)
            
            # 리뷰 별점
            star = review.find_element(By.CSS_SELECTOR, "div.star-grade > strong")
            style = star.get_attribute("style")
            match = re.search(r"width:\s*([0-9]+)%", style)
    
            if match :
                star_percent = float(match.group(1))
                review_rating = star_percent / 20
            else :
                review_rating = None
    
            # 리뷰 상세 평가 데이터 저장을 위한 딕셔너리 정의
            evaluations = dict() # | {}
    
            # 리뷰 상세 평가
            try :
                evaluation_items = review.find_elements(By.CSS_SELECTOR, "ul.product_review_evaluation > li")
                for evaluation in evaluation_items :
                    evaluation_type = evaluation.find_element(By.CSS_SELECTOR, "strong").text.strip()
                    evaluation_value = evaluation.find_element(By.CSS_SELECTOR, "em").text.strip()
                    evaluations[evaluation_type] = evaluation_value
            except :
                continue
    
            # 리뷰 이미지 저장용 리스트
            review_images = list()
    
            # 리뷰 이미지
            images = review.find_elements(By.CSS_SELECTOR, "ul.pdt_review_photo img")
        
            for image in images :
                image_url = image.get_attribute("src")
                review_images.append(image_url)

            review_data = {
                "product_id": product["product_id"],
                "brand_name": product["brand_name"],
                "product_name": product["product_name"],
                "product_url": product["product_url"],
                "review_page": page_no,
                "review_no": review_no,
                "reviewer": reviewer,
                "review_date": review_date,
                "purchase_option": purchase_option,
                "review_text": review_text,
                "review_rating": review_rating,
                "evaluations": evaluations,
                "review_images": review_images                
            }

            all_reviews.append(review_data)
            
product_df= pd.DataFrame(products)
review_df = pd.DataFrame(all_reviews)            
print("4. ✅ wconcept 전체리뷰수집 완료")

In [ ]:
import re
import requests
import pandas as pd

NAVER_CLIENT_ID = ""
NAVER_CLIENT_SECRET = ""

NAVER_BLOG_API_URL = "https://naverapihub.apigw.ntruss.com/search/v1/blog"

headers = {
    "X-NCP-APIGW-API-KEY-ID": NAVER_CLIENT_ID,
    "X-NCP-APIGW-API-KEY": NAVER_CLIENT_SECRET
}

# 유틸리티 함수
def clean_text(text) :
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"&amp;", "", text)
    return text.strip()


brand_names = product_df["brand_name"].dropna().drop_duplicates().tolist()
# brand_names = brand_names[:1]

print("검색할 전체 브랜드 수:", len(brand_names))

# 블로그 컨텐츠 중 취합할 요소만 찾아서 컨텐츠별 저장할 딕셔너리를 모아놓기위한 용도의 리스트
all_blog_posts = list() # []

for brand_index, brand_name in enumerate(brand_names, 1) :
    print( f"[{brand_index}/{len(brand_names)}] {brand_name} 블로그 검색 중...")
    params = {
        "query": brand_name,
        "display": 10,
        "start": 1,
        "sort": "date",
        "format": "json"
    }
    try :
        response = requests.get(NAVER_BLOG_API_URL, headers=headers, params=params, timeout=10)

        if response.status_code == 200 :
            data = response.json()
            items = data.get("items", [])

            for item in items :
                blog_data = {
                    "search_brand": brand_name,
                    "title": clean_text(item.get("title", "")),
                    "link": item.get("link", ""),
                    "description": clean_text(item.get("description", "")),
                    "bloggername": item.get("bloggername", ""),
                    "bloggerlink": item.get("bloggerlink", ""),
                    "postdate": item.get("postdate", "")
                }
                
                all_blog_posts.append(blog_data)
                
    except Exception as e :
        print(f"API 호출 중 오류 발생 : {e}")
        
blog_df = pd.DataFrame(all_blog_posts)

blog_df["postdate"]= pd.to_datetime(blog_df["postdate"], format="%Y%m%d", errors="coerce") #NaT = Not a Time

print("수집 완료")
print("브랜드 수:", len(brand_names))
print("블로그 게시물 수:", len(blog_df))

In [ ]:
# 위 파일이 정상적으로 실행되었다는 전제하에 아래 구문 시작!
# product_df 상품 200개 데이터
# .csv 파일 포맷(형태) 저장

# 상품 데이터를 csv
product_df.to_csv(
    "wconcept_products_20260915.csv",
    index=False,
    encoding="utf-8-sig"
)

# 리뷰 데이터를 csv
review_save_df = review_df.copy()

review_save_df["evaluations"] = (
    review_save_df["evaluations"]
    .apply(
        lambda x: 
        json.dumps(
            x, ensure_ascii=False
        ) if isinstance(x, dict)
        else "{}"
    )
)

review_save_df["review_images"] = (
    review_save_df["review_images"]
    .apply(
        lambda x: 
        json.dumps(
            x, ensure_ascii=False
        ) if isinstance(x, list)
        else "[]"
    )
)

review_save_df.to_csv(
    "wconcept_reviews_20260915.csv",
    index=False,
    encoding="utf-8-sig"
)

# def func1(a, b) :
#     return a + b

# 람다식 = 익명함수

# if num = 1:
#     print(1)
# else :
#     print(2)

# 3항 조건 연산식

In [1]:
# csv 파일 포맷을 불러오는 방법

import pandas as pd
import json

# 상품 데이터 복원
product_df = pd.read_csv(
    "./final/wconcept_products_20260915.csv"
)

# 리뷰 데이터 복원
review_df = pd.read_csv(
    "./final/wconcept_reviews_20260915.csv"
)

review_df["evaluations"] = (
    review_df["evaluations"]
    .apply(
        lambda x: 
        json.loads(x)
        if pd.notna(x)
        else {}
    )
)

review_df["review_images"] = (
    review_df["review_images"]
    .apply(
        lambda x: 
        json.loads(x)
        if pd.notna(x)
        else []
    )
)

print(product_df) # 상품정보 200개
print(product_df) # 리뷰정보 (상품당)

FileNotFoundError: [Errno 2] No such file or directory: './final/wconcept_products_20260915.csv'

In [ ]:
# 네이버 블로그 API 데이터 CSV 저장

# csv 파일 포맷으로 변환.저장 > 특정 폴더 (어떤 경로 안에 존재) 안에 저장!!!
import os # opertating system : 현재 작업중인 경로, 폴더위치 등의 데이터 관리.수집
from datetime import datetime # 로컬 컴퓨터의 현재 시간정보! -> 어떤 값을 저장하고자 할 때, 저장되는 시점에 맞춰서 파일명을 작성할 수 있음

SAVE_DIR = "./wconcept_data"

os.makedirs(SAVE_DIR, exist_ok=True)
collected_at = datetime.now().strftime("%Y%m%d_%H%M%S")
blog_csv_file = f"{SAVE_DIR}/wconcept_blogs_{collected_at}.csv"

blog_df.to_csv(blog_csv_file, index=False, encoding="utf-8-sig")